# 04 — Main Evaluation: Qwen2.5-Omni-7B (120 samples × S0–S5)

Runs the canonical evaluation. Results land in `results/raw_predictions/qwen/`.

**Runtime budget:** ~2hr on H100. Per-stage estimate from the pilot (`results/raw_predictions/pilot/_timings.json`):
- S0 (text-only): negligible
- S1, S2, S3, S4, S5: ~12s/sample × 120 ≈ 24min each

Each stage is checkpointed every 20 samples, so a Colab disconnect doesn't lose work — re-running this notebook resumes from the last checkpoint.


In [ ]:
import os, sys, json, time
REPO = '/content/omnimodel-research'
if os.path.exists(REPO):
    %cd $REPO
    sys.path.insert(0, REPO)

with open('data/eval_samples_clean.json') as f:
    samples = json.load(f)
print(f'Eval samples: {len(samples)}')


In [ ]:
from src.model_utils import OmniModelWrapper, TextOnlyModelWrapper
from src.prompts import STAGE_BUILDERS
from src.parse_utils import parse_answer_confidence, parse_attribution, parse_reason
from src.transcript_utils import load_transcript

print('Loading models...')
omni = OmniModelWrapper('Qwen/Qwen2.5-Omni-7B')
text_only = TextOnlyModelWrapper('Qwen/Qwen2.5-1.5B-Instruct')
print('Models loaded.')


In [ ]:
def stage_inputs(stage, sample):
    vid = sample['video_id']
    paths = {
        'video':      f'data/videos/{vid}.mp4',
        'audio':      f'data/audio/{vid}.wav',
        'silent':     f'data/silent_video/{vid}_silent.mp4',
        'transcript': f'data/transcripts/{vid}.txt',
        # Mismatched transcript is keyed by qa_id (a single video may have
        # multiple QAs that need DIFFERENT mismatched donors)
        'mismatched': f'data/transcripts_mismatched/{sample["qa_id"]}.txt',
    }
    routes = {
        'S0': dict(video_path=None, audio_path=None, extra={}),
        'S1': dict(video_path=None, audio_path=paths['audio'], extra={}),
        'S2': dict(video_path=paths['silent'], audio_path=None, extra={}),
        'S3': dict(video_path=paths['video'], audio_path=paths['audio'], extra={}),
        'S4': dict(video_path=paths['video'], audio_path=paths['audio'],
                   extra={'transcript': load_transcript(paths['transcript'])}),
        'S5': dict(video_path=paths['video'], audio_path=paths['audio'],
                   extra={'mismatched_transcript': load_transcript(paths['mismatched'])}),
    }
    return routes[stage]

def run_one(stage, sample, model):
    inp = stage_inputs(stage, sample)
    prompt = STAGE_BUILDERS[stage](sample['question'], sample['options'], **inp['extra'])
    resp = model.generate(prompt, video_path=inp['video_path'], audio_path=inp['audio_path'])
    answer, conf = parse_answer_confidence(resp.text)
    attr = parse_attribution(resp.text) if stage == 'S3' else None
    reason = parse_reason(resp.text) if stage == 'S3' else None
    return {
        'video_id': sample['video_id'], 'task_type': sample['task_type'], 'stage': stage,
        'ground_truth': sample['answer'], 'predicted_answer': answer,
        'verbalized_confidence': conf, 'answer_logprobs': resp.answer_logprobs,
        'attribution': attr, 'attribution_reason': reason, 'raw_response': resp.text,
    }

OUT_DIR = 'results/raw_predictions/qwen'
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
def run_stage(stage, samples, model, checkpoint_every=20):
    out_path = f'{OUT_DIR}/{stage}.json'
    done = []
    if os.path.exists(out_path):
        with open(out_path) as f:
            done = json.load(f)
        print(f'  {stage}: resuming with {len(done)} cached records')
    done_ids = {r['qa_id'] for r in done}
    todo = [s for s in samples if s['qa_id'] not in done_ids]
    if not todo:
        print(f'  {stage}: already complete')
        return done

    t0 = time.time()
    records = list(done)
    for i, s in enumerate(todo):
        try:
            rec = run_one(stage, s, model)
        except Exception as e:
            rec = {'video_id': s['video_id'], 'task_type': s['task_type'], 'stage': stage,
                   'ground_truth': s['answer'], 'predicted_answer': None,
                   'verbalized_confidence': None, 'answer_logprobs': None,
                   'attribution': None, 'attribution_reason': None,
                   'raw_response': f'<ERROR: {e}>'}
        records.append(rec)
        if (i + 1) % checkpoint_every == 0 or (i + 1) == len(todo):
            with open(out_path, 'w') as f:
                json.dump(records, f, indent=2)
            elapsed = time.time() - t0
            done_n = len(records)
            print(f'  {stage}: {done_n}/{len(samples)}  ({elapsed:.0f}s elapsed, {elapsed/(i+1):.1f}s/sample)')
    return records

STAGES = ['S0', 'S1', 'S2', 'S3', 'S4', 'S5']
all_results = {}
for stage in STAGES:
    print(f'\n=== Running {stage} ===')
    model = text_only if stage == 'S0' else omni
    all_results[stage] = run_stage(stage, samples, model)
print('\n=== Main eval complete ===')


In [ ]:
# Quick sanity scan
from src.metrics import accuracy_per_task
for stage in STAGES:
    with open(f'{OUT_DIR}/{stage}.json') as f:
        recs = json.load(f)
    acc = accuracy_per_task(recs)
    parse_rate = sum(1 for r in recs if r['predicted_answer'] is not None) / len(recs)
    print(f'{stage}: parse={parse_rate:.0%}  overall_acc={acc.get("OVERALL", 0):.2f}  per-task={ {t: f"{v:.2f}" for t, v in acc.items() if t != "OVERALL"} }')


**Done.** Proceed to `06_analysis.ipynb` for metric computation, or `05_main_eval_gemma.ipynb` for cross-model comparison.